# Chapter 4 · Tools and Interoperability
**Mastering Agentic AI** — Manning Publications

## What this notebook covers
- How tool calling works internally (3 stages)
- Five principles for production-ready tools
- Building a resilient `get_nutrition` tool
- MCP: Host, Client, Server architecture
- Real MCP client/server implementation with the official SDK
- Composio and tooling ecosystems
- Agent Skills + Tools working together


## 4.0 · Setup

In [ ]:
# !pip install openai mcp requests python-dotenv composio composio_crewai crewai langchain-openai
import os, json, logging, sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('.env'), override=False)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

from openai import OpenAI

try:
    import anyio
    from mcp import ClientSession, StdioServerParameters, stdio_client
    from mcp.server import FastMCP
    from mcp.shared.memory import create_connected_server_and_client_session
except ImportError:
    anyio = None
    ClientSession = None
    StdioServerParameters = None
    stdio_client = None
    FastMCP = None
    create_connected_server_and_client_session = None

try:
    from composio import Composio
    from composio_crewai import CrewAIProvider
except ImportError:
    Composio = None
    CrewAIProvider = None

try:
    from crewai import Agent, Task, Crew
except ImportError:
    Agent = None
    Task = None
    Crew = None

try:
    from langchain_openai import ChatOpenAI
except ImportError:
    ChatOpenAI = None

client = OpenAI()
MODEL = 'gpt-4.1-nano'
print('Ready')


## 4.1 · Tool Calling Internals

Three stages:
1. **Configuration** — inject tool manifest into context (name + schema + docstring)
2. **Invocation** — model emits structured JSON intent
3. **Execution** — YOUR code runs the function; result returned to model

The model never executes code. It describes intent. You decide whether to run it.

In [ ]:
# Stage 2 example: what the model emits
INVOCATION_EXAMPLE = {
    'tool_name': 'get_nutrition',
    'arguments': {'food_item': '1 banana'}
}
print('Model emits INTENT:')
print(json.dumps(INVOCATION_EXAMPLE, indent=2))
print()
print('Key insight: the model NEVER calls the function.')
print('YOUR orchestrator calls the function and returns the result.')

## 4.2 · Five Principles for Production-Ready Tools

In [ ]:
import requests

NUTRITION_API_URL = 'https://trackapi.nutritionix.com/v2/natural/nutrients'
API_KEY = os.getenv('NUTRITIONIX_API_KEY', 'DEMO')
APP_ID  = os.getenv('NUTRITIONIX_APP_ID',  'DEMO')

def get_nutrition(food_item: str) -> str:
    # Principle 1: Docstring is the LLM's user interface
    '''
    Fetches nutritional information for a single food item from NutritionIX.
    For raw ingredients only (e.g., '1 banana', '150g chicken breast').
    Not suitable for complex meals or restaurant dishes.
    Returns: JSON with status=success|not_found|error and nutritional data.
    '''
    logging.info(f"Tool called: food_item={food_item!r}")  # Principle 5: Log

    if not food_item or not isinstance(food_item, str):    # Principle 3: Validate
        return json.dumps({'status': 'error', 'message': 'food_item must be non-empty string'})

    headers = {'x-app-id': APP_ID, 'x-app-key': API_KEY, 'Content-Type': 'application/json'}
    body    = {'query': food_item.lower().strip()}

    try:                                                   # Principle 4: Build for failure
        r = requests.post(NUTRITION_API_URL, headers=headers, json=body, timeout=10)
        r.raise_for_status()
        data = r.json()
        if not data.get('foods'):
            return json.dumps({'status': 'not_found', 'message': f'{food_item!r} not found'})
        f = data['foods'][0]
        return json.dumps({'status': 'success', 'data': {
            'item': f.get('food_name'), 'calories': f.get('nf_calories'),
            'protein_g': f.get('nf_protein'), 'carbs_g': f.get('nf_total_carbohydrate')
        }})
    except requests.exceptions.Timeout:
        return json.dumps({'status': 'error', 'message': 'API timed out'})
    except Exception as e:
        logging.critical(f'Unexpected: {e}', exc_info=True)
        return json.dumps({'status': 'error', 'message': 'Unexpected internal error'})

# Mock for demos without API keys
NUTRITION_DB = {'apple': {'calories':95,'protein_g':0.5,'carbs_g':25},
                'oats': {'calories':150,'protein_g':5.0,'carbs_g':27},
                'salmon': {'calories':208,'protein_g':28.0,'carbs_g':0},
                'banana': {'calories':89,'protein_g':1.1,'carbs_g':23},
                'chicken breast': {'calories':165,'protein_g':31.0,'carbs_g':0}}

def get_nutrition_mock(food_item: str) -> str:
    key = food_item.strip().lower()
    data = NUTRITION_DB.get(key)
    if data:
        return json.dumps({'status': 'success', 'data': {'item': key, **data}})
    return json.dumps({'status': 'not_found', 'message': f'{food_item!r} not in mock DB'})

print(get_nutrition_mock('banana'))
print(get_nutrition_mock('pizza'))  # structured not_found — not a crash

## 4.3 · MCP Architecture

In [ ]:
print('''
MCP — Model Context Protocol (Anthropic, 2024)

Three components:
  Host   — the AI application; manages tool discovery and routing
  Client — one per server; translates host requests to protocol calls
  Server — standalone program exposing tools via JSON-RPC 2.0

Flow: LLM → intent → Host → Client → Server → executes → result → LLM

Before MCP: framework-specific integrations, rebuilt for every ecosystem
After MCP:  write once, use everywhere (CrewAI, LangChain, Cursor, Claude Desktop)

Think of MCP as USB-C for agent tools.
''')

## 4.4 · MCP Client (official SDK)

The real Python MCP client performs an initialization handshake before it can
list tools or call them. The client does **not** send raw `tools/list`
immediately. Instead, it connects, initializes the session, then uses the
protocol methods exposed by the SDK.


In [ ]:
MCP_SERVER_FILENAME = 'diet_coach_mcp_server.py'


def _require_mcp_sdk():
    if anyio is None or ClientSession is None or StdioServerParameters is None or stdio_client is None:
        raise RuntimeError('Install MCP SDK first: pip install mcp')


def _tool_to_dict(tool):
    return {
        'name': tool.name,
        'description': tool.description,
        'input_schema': tool.inputSchema,
    }


async def list_mcp_tools_async(server_script_path: str):
    _require_mcp_sdk()
    server = StdioServerParameters(
        command=sys.executable,
        args=[server_script_path],
        cwd=str(Path(server_script_path).parent),
    )
    async with stdio_client(server) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.list_tools()
            return [_tool_to_dict(tool) for tool in result.tools]


async def call_mcp_tool_async(server_script_path: str, tool_name: str, arguments: dict):
    _require_mcp_sdk()
    server = StdioServerParameters(
        command=sys.executable,
        args=[server_script_path],
        cwd=str(Path(server_script_path).parent),
    )
    async with stdio_client(server) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
            text_blocks = [block.text for block in result.content if getattr(block, 'type', None) == 'text']
            return {
                'is_error': result.isError,
                'content': text_blocks,
                'structured_content': result.structuredContent,
            }


print('Official MCP client helpers ready.')
print('Usage:')
print('  tools = anyio.run(list_mcp_tools_async, "diet_coach_mcp_server.py")')
print('  result = anyio.run(call_mcp_tool_async, "diet_coach_mcp_server.py", "lookup_nutrition", {"food_item": "oats"})')


## 4.5 · MCP Server (real FastMCP server)

We now expose the nutrition lookup capability through a real MCP server. This
keeps the tool reusable across MCP-compatible hosts while preserving the same
nutrition domain we used earlier in the chapter.


In [ ]:
NUTRITION_DB = {
    'apple':          {'calories': 95,  'protein_g': 0.5,  'carbs_g': 25, 'fat_g': 0.3, 'fibre_g': 4.4},
    'chicken breast': {'calories': 165, 'protein_g': 31.0, 'carbs_g': 0,  'fat_g': 3.6, 'fibre_g': 0},
    'brown rice':     {'calories': 216, 'protein_g': 5.0,  'carbs_g': 45, 'fat_g': 1.8, 'fibre_g': 3.5},
    'broccoli':       {'calories': 55,  'protein_g': 3.7,  'carbs_g': 11, 'fat_g': 0.6, 'fibre_g': 5.1},
    'salmon':         {'calories': 208, 'protein_g': 28.0, 'carbs_g': 0,  'fat_g': 10,  'fibre_g': 0},
    'oats':           {'calories': 150, 'protein_g': 5.0,  'carbs_g': 27, 'fat_g': 2.5, 'fibre_g': 4.0},
}


def build_mcp_server():
    _require_mcp_sdk()
    server = FastMCP(
        'diet-coach-nutrition',
        instructions='Nutrition lookup tools for the AI Diet Coach.',
    )

    @server.tool(
        name='lookup_nutrition',
        description='Return macro-nutrient data for a food item per 100 g serving.',
        structured_output=True,
    )
    def lookup_nutrition_mcp(food_item: str) -> dict:
        key = food_item.strip().lower()
        data = NUTRITION_DB.get(key)
        if data:
            return {'food': key, **data}

        matches = [k for k in NUTRITION_DB if key in k or k in key]
        if matches:
            match = matches[0]
            return {'food': match, 'note': f"closest match for '{food_item}'", **NUTRITION_DB[match]}

        return {'error': f"'{food_item}' not found"}

    return server


async def verify_mcp_locally_async():
    _require_mcp_sdk()
    server = build_mcp_server()
    async with create_connected_server_and_client_session(server) as session:
        tools = await session.list_tools()
        result = await session.call_tool('lookup_nutrition', {'food_item': 'oats'})
        text_blocks = [block.text for block in result.content if getattr(block, 'type', None) == 'text']
        return {
            'tools': [_tool_to_dict(tool) for tool in tools.tools],
            'tool_call': {
                'is_error': result.isError,
                'content': text_blocks,
                'structured_content': result.structuredContent,
            },
        }


MCP_SERVER_CODE = '''#!/usr/bin/env python3
from mcp.server import FastMCP

NUTRITION_DB = {
    "apple":          {"calories": 95,  "protein_g": 0.5,  "carbs_g": 25, "fat_g": 0.3, "fibre_g": 4.4},
    "chicken breast": {"calories": 165, "protein_g": 31.0, "carbs_g": 0,  "fat_g": 3.6, "fibre_g": 0},
    "brown rice":     {"calories": 216, "protein_g": 5.0,  "carbs_g": 45, "fat_g": 1.8, "fibre_g": 3.5},
    "broccoli":       {"calories": 55,  "protein_g": 3.7,  "carbs_g": 11, "fat_g": 0.6, "fibre_g": 5.1},
    "salmon":         {"calories": 208, "protein_g": 28.0, "carbs_g": 0,  "fat_g": 10,  "fibre_g": 0},
    "oats":           {"calories": 150, "protein_g": 5.0,  "carbs_g": 27, "fat_g": 2.5, "fibre_g": 4.0},
}

server = FastMCP('diet-coach-nutrition', instructions='Nutrition lookup tools for the AI Diet Coach.')

@server.tool(name='lookup_nutrition', description='Return macro-nutrient data for a food item per 100 g serving.', structured_output=True)
def lookup_nutrition(food_item: str) -> dict:
    key = food_item.strip().lower()
    data = NUTRITION_DB.get(key)
    if data:
        return {'food': key, **data}
    matches = [k for k in NUTRITION_DB if key in k or k in key]
    if matches:
        match = matches[0]
        return {'food': match, 'note': f"closest match for '{food_item}'", **NUTRITION_DB[match]}
    return {'error': f"'{food_item}' not found"}

if __name__ == '__main__':
    server.run(transport='stdio')
'''

server_output_path = Path('chapter_04_tools') / MCP_SERVER_FILENAME if Path('chapter_04_tools').exists() else Path(MCP_SERVER_FILENAME)
server_output_path.write_text(MCP_SERVER_CODE)
print(f'MCP server file written to: {server_output_path.resolve()}')

if anyio is not None:
    verification = anyio.run(verify_mcp_locally_async)
    print(json.dumps(verification, indent=2))
else:
    print('Install mcp to run the local MCP verification demo.')


## 4.6 · Composio — Updated Tooling Ecosystem Pattern

Composio is not a replacement for MCP. It is a tooling ecosystem that provides
managed integrations, authentication, and framework-formatted toolkits.

A useful mental model:
- MCP standardises interoperability
- Composio accelerates access to common integrations
- Custom tools still matter for domain-specific logic such as nutrition lookup


In [ ]:
COMPOSIO_EXPLAINER = '''
COMPOSIO — Managed Tooling Ecosystem

Typical workflow:
  1. Create or reuse an auth config for a toolkit (e.g. GitHub)
  2. Link a user account to that toolkit
  3. Fetch provider-formatted tools for your framework
  4. Give those tools to an agent
'''


def _require_composio_sdk():
    if Composio is None or CrewAIProvider is None:
        raise RuntimeError('Install Composio first: pip install composio composio_crewai')


def create_composio_github_auth_link(user_id: str, auth_config_id: str, callback_url: str | None = None) -> str:
    _require_composio_sdk()
    composio = Composio()
    request = composio.connected_accounts.link(
        user_id=user_id,
        auth_config_id=auth_config_id,
        callback_url=callback_url,
    )
    return request.redirect_url


def get_composio_github_tools(user_id: str = 'default'):
    _require_composio_sdk()
    composio = Composio(provider=CrewAIProvider())
    return composio.tools.get(user_id=user_id, toolkits=['GITHUB'])


def build_composio_github_crew(user_id: str = 'default'):
    _require_composio_sdk()
    if Agent is None or Task is None or Crew is None:
        raise RuntimeError('Install CrewAI first: pip install crewai')
    if ChatOpenAI is None:
        raise RuntimeError('Install langchain-openai first: pip install langchain-openai')

    github_tools = get_composio_github_tools(user_id=user_id)
    llm = ChatOpenAI()

    github_agent = Agent(
        role='GitHub Research Agent',
        goal='Understand recent activity in a GitHub repository',
        backstory=(
            'An expert developer agent that inspects repositories, '
            'reads commits and pull requests, and summarises changes.'
        ),
        verbose=True,
        tools=github_tools,
        llm=llm,
    )

    task = Task(
        description='Inspect the latest activity in a GitHub repository and produce a short summary of what changed.',
        expected_output='A concise summary of recent repository activity.',
        agent=github_agent,
    )

    return Crew(agents=[github_agent], tasks=[task])


print(COMPOSIO_EXPLAINER)
if Composio is not None and CrewAIProvider is not None and os.getenv('COMPOSIO_API_KEY'):
    github_user_id = os.getenv('COMPOSIO_USER_ID', 'default')
    print('Composio SDK detected.')
    print(f"Configured user_id: {github_user_id}")
    print('Use get_composio_github_tools(...) after connecting a GitHub account in Composio.')
else:
    print('Skipping live Composio demo. Install composio + composio_crewai and set COMPOSIO_API_KEY.')


## 4.7 · Agent Skills + Tools: The Full Pattern

In [ ]:
SKILL_CONTENT = '''# Nutrition Assessment Skill
Step 1 — Establish Baseline: eating pattern, restrictions, goals, activity.
Step 2 — Identify Deficits: protein (<0.8g/kg), fibre (<25g/day), micronutrients.
Step 3 — Prioritise: high-impact, low-effort first.
Step 4 — One Measurable Goal: concrete and time-bound. One goal only.'''

TOOLS_FOR_AGENT = [
    {"type": "function", "function": {
        "name": "lookup_nutrition",
        "description": "Return macro-nutrient data for a food item per 100g.",
        "parameters": {"type": "object", "properties": {"food_item": {"type": "string"}}, "required": ["food_item"]}}},
    {"type": "function", "function": {
        "name": "suggest_meal",
        "description": "Suggest a meal matching a dietary goal (e.g., high protein, low carb).",
        "parameters": {"type": "object", "properties": {"goal": {"type": "string"}, "max_calories": {"type": "integer"}}, "required": ["goal"]}}},
]

TOOL_FNS = {
    'lookup_nutrition': get_nutrition_mock,
    'suggest_meal': lambda goal, max_calories=600: json.dumps({'suggestions': ['oats','salmon','chicken breast']}),
}

def run_skill_guided_agent(user_message):
    system = (f'You are an AI Diet Coach.\n[Skill]\n{SKILL_CONTENT}\n'
              'Use tools for data. Apply skill protocol for full assessments.')
    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': user_message},
    ]

    while True:
        response = client.chat.completions.create(
            model=MODEL, max_tokens=1024,
            tools=TOOLS_FOR_AGENT, messages=messages)
        choice = response.choices[0]
        messages.append(choice.message)

        if choice.finish_reason == 'stop':
            return choice.message.content or ''

        if choice.finish_reason != 'tool_calls':
            break

        for tc in choice.message.tool_calls:
            fn = TOOL_FNS.get(tc.function.name)
            args = json.loads(tc.function.arguments)
            result = fn(**args) if fn else json.dumps({'error': 'unknown'})
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})

    return '[loop ended unexpectedly]'

answer = run_skill_guided_agent('I want high protein dinner under 500 calories. What should I eat?')
print('Coach:', answer)

## 4.8 · Chapter Summary

| Concept | Key insight |
|---|---|
| Tool calling internals | 3 stages: configure → invoke → execute (you are the executor) |
| Production tool design | 5 principles: docstring, atomic, defensive, failure-ready, logged |
| MCP architecture | Decouples reasoning from execution across frameworks |
| MCP client | Official SDK client performs handshake, discovery, then tool calls |
| MCP server | FastMCP wraps tools as reusable MCP capabilities |
| Composio | Tooling ecosystem: managed toolkits over repeated API wiring |
| Skill + Tool | Skills = HOW to think; Tools = WHAT to say |

**Next:** Chapter 5 — Memory: agents that remember across sessions.

---
*Mastering Agentic AI · Manning Publications*
